In [1]:
import carla
import math
import random
import time
import numpy as np
import cv2

client = carla.Client('localhost', 2000)
client.set_timeout(10.0)

world = client.get_world()
traffic_manager = client.get_trafficmanager()
traffic_manager.set_synchronous_mode(True)
settings = world.get_settings()
settings.synchronous_mode = True
RAW_FPS = 30
settings.fixed_delta_seconds = 1.0 / RAW_FPS  # 30 fps raw video; dataset samples are subsampled to 5 fps
world.apply_settings(settings)

bp_lib = world.get_blueprint_library()
spawn_points = world.get_map().get_spawn_points()

print("Setup cell ran succesfully")

Setup cell ran succesfully


In [2]:
vehicle_bp = bp_lib.find('vehicle.tesla.model3')
vehicle = world.try_spawn_actor(vehicle_bp, random.choice(spawn_points))
#vehicle.show_debug_telemetry(True)
world.tick()  # sync mode: advance one step so the vehicle actually appears

spectator = world.get_spectator()

IMAGE_W = 1600
IMAGE_H = 900

def make_camera_bp(fov):
    bp = bp_lib.find('sensor.camera.rgb')
    bp.set_attribute('image_size_x', str(IMAGE_W))
    bp.set_attribute('image_size_y', str(IMAGE_H))
    bp.set_attribute('fov', str(fov))
    return bp

# 6 camera suite setup: (name, transform, fov)
# note: back_right uses a wider 110 fov, the rest use 70
camera_specs = [
    ('front',       carla.Transform(carla.Location(z=1.5)),                            70),   # front middle
    ('front_left',  carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=-55)),   70),   # front left
    ('front_right', carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+55)),   70),   # front right
    ('back',        carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+180)),  70),   # back middle
    ('back_left',   carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=-120)),  70),   # back left
    ('back_right',  carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+120)),  110),  # back right
]

In [3]:
# spawn all 6 cameras at the same time, attached to the vehicle
cameras = {}
for name, trans, fov in camera_specs:
    cameras[name] = world.spawn_actor(make_camera_bp(fov), trans, attach_to=vehicle)

print('spawned', len(cameras), 'cameras:', list(cameras))

# park the spectator on the front camera so the CARLA window has a useful view
spectator.set_transform(cameras['front'].get_transform())
world.tick()  # sync mode: advance one step so the cameras register and the spectator move renders

spawned 6 cameras: ['front', 'front_left', 'front_right', 'back', 'back_left', 'back_right']


376273

In [4]:
import os

# ---- dataset layout ---------------------------------------------------------
# dataset/<RUN_ID>/
#   FRONT/000000.png ... BACK_RIGHT/000000.png   raw 30 fps frames, contiguous index
#   calibration.json                              camera intrinsics + extrinsics
#   output.json                                   one entry per 3 s clip sample
# Each recording lives in its own RUN_ID folder so sessions never mix. The loop
# cell clears this run's frames on start (so disk == buffer); bump RUN_ID to keep
# an older run instead of overwriting it.
OUTPUT_DIR = 'dataset'
RUN_ID = 'run02'
RUN_DIR = os.path.join(OUTPUT_DIR, RUN_ID)
folder_for = {name: name.upper() for name in cameras}   # 'front' -> 'FRONT', etc.
for folder in folder_for.values():
    os.makedirs(os.path.join(RUN_DIR, folder), exist_ok=True)

# latest frame per camera, kept only for the opencv preview windows
camera_data = {name: np.zeros((IMAGE_H, IMAGE_W, 4), dtype=np.uint8) for name in cameras}

In [5]:
import json

# ---- camera calibration (written once; constant for the whole run) ----------
# intrinsics depend only on resolution + fov, and the camera->vehicle extrinsic is
# the static mounting transform. camera->world for any frame is then:
#   T_cam2world = T_vehicle2world(frame) @ T_cam2vehicle
# where T_vehicle2world comes from the ego pose (x,y,z,pitch,yaw,roll).
# all values use CARLA's convention: x forward, y right, z up, metres/degrees.

def intrinsic_matrix(width, height, fov_deg):
    f = width / (2.0 * math.tan(math.radians(fov_deg) / 2.0))  # square pixels, horizontal fov
    cx, cy = width / 2.0, height / 2.0
    return [[f,   0.0, cx],
            [0.0, f,   cy],
            [0.0, 0.0, 1.0]]

calibration = {}
for name, trans, fov in camera_specs:
    loc, rot = trans.location, trans.rotation
    calibration[folder_for[name]] = {
        'image_size': [IMAGE_W, IMAGE_H],
        'fov': fov,
        'intrinsic': intrinsic_matrix(IMAGE_W, IMAGE_H, fov),
        # static 4x4 camera->vehicle extrinsic (row-major homogeneous matrix)
        'extrinsic_cam_to_vehicle': trans.get_matrix(),
        # same mounting in human-readable form
        'mounting': {
            'location': [loc.x, loc.y, loc.z],
            'rotation': {'pitch': rot.pitch, 'yaw': rot.yaw, 'roll': rot.roll},
        },
    }

with open(os.path.join(RUN_DIR, 'calibration.json'), 'w') as cal_file:
    json.dump(calibration, cal_file, indent=2)
print('wrote', os.path.join(RUN_DIR, 'calibration.json'))

wrote dataset/run01/calibration.json


In [6]:
import importlib
import dataset_builder
importlib.reload(dataset_builder)  # pick up edits to dataset_builder.py without restarting the kernel

carla_map = world.get_map()

# ---- map zone via the CARLA map API (instruction-document item 3) ------------
# The document's hand-drawn lab polygons are replaced by CARLA's own road graph.
# For the ego location we look up the nearest driving waypoint and report:
#   zone            'intersection' if in a junction, else 'lane'
#   road_segment    road/lane (or junction) identifiers
#   nearest_landmark first upcoming sign/light, if any
#   distance_to_next_turn_m  metres to the next junction along the road
def _distance_to_next_junction(wp, max_dist=80.0, step=2.0):
    travelled, cur = 0.0, wp
    while travelled < max_dist:
        nxt = cur.next(step)
        if not nxt:
            return None
        cur = nxt[0]
        travelled += step
        if cur.is_junction:
            return round(travelled, 2)
    return None  # no junction within max_dist

def get_map_context(carla_map, location):
    wp = carla_map.get_waypoint(location, project_to_road=True,
                                lane_type=carla.LaneType.Driving)
    if wp is None:
        return {'zone': 'unknown', 'road_segment': None,
                'nearest_landmark': None, 'distance_to_next_turn_m': None}
    if wp.is_junction:
        junction = wp.get_junction()
        zone = 'intersection'
        road_segment = f'junction_{junction.id}' if junction else 'junction'
        dist_to_turn = 0.0
    else:
        zone = 'lane'
        road_segment = f'road_{wp.road_id}_lane_{wp.lane_id}'
        dist_to_turn = _distance_to_next_junction(wp)
    nearest_landmark = None
    try:
        lms = wp.get_landmarks(80.0, stop_at_junction=False)
        if lms:
            nearest_landmark = lms[0].name or str(lms[0].type)
    except RuntimeError:
        pass
    return {'zone': zone, 'road_segment': road_segment,
            'nearest_landmark': nearest_landmark,
            'distance_to_next_turn_m': dist_to_turn}

print('map-context + dataset-builder helpers ready')

map-context + dataset-builder helpers ready


In [7]:
import queue

# one queue per camera. in synchronous mode every world.tick() pushes exactly one
# image per camera, and all six images share the same frame id. matching on that
# frame id in the capture loop is what bundles a perfectly synchronized snapshot.
image_queues = {name: queue.Queue() for name in cameras}
for name, cam in cameras.items():
    cam.listen(image_queues[name].put)

In [8]:
import shutil

# bind to the same Traffic Manager that was put into synchronous mode in the setup cell
vehicle.set_autopilot(True, traffic_manager.get_port())

# one window per camera, tiled in a 3x2 grid that mirrors the physical layout
# so the windows don't open stacked on top of each other.
#   row 0:  Front Left | Front | Front Right
#   row 1:  Back Left  | Back  | Back Right
titles = {
    'front_left': 'Front Left',  'front': 'Front',  'front_right': 'Front Right',
    'back_left':  'Back Left',   'back':  'Back',    'back_right':  'Back Right',
}
grid = {
    'front_left': (0, 0),  'front': (1, 0),  'front_right': (2, 0),
    'back_left':  (0, 1),  'back':  (1, 1),  'back_right':  (2, 1),
}

THUMB_W, THUMB_H = 480, 270   # display size per window (16:9, scaled down from 1600x900)
GAP_X, GAP_Y = 12, 48         # spacing between windows; GAP_Y leaves room for title bars

for name, (col, row) in grid.items():
    win = titles[name]
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, THUMB_W, THUMB_H)
    cv2.moveWindow(win, col * (THUMB_W + GAP_X), row * (THUMB_H + GAP_Y))

# fresh recording session: wipe THIS run's camera folders so the contiguous frame
# index restarts at 0 and the saved frames match frames_log exactly (disk == buffer).
for folder in folder_for.values():
    cam_dir = os.path.join(RUN_DIR, folder)
    shutil.rmtree(cam_dir, ignore_errors=True)
    os.makedirs(cam_dir, exist_ok=True)

recording = False
seq = 0           # contiguous frame index (00000, 00001, ...) and the sync key
frames_log = []   # in-memory per-frame buffer; the build cell slices it into 3 s clips
print("press 'r' to toggle recording, 'q' to quit")

while True:
    # in synchronous mode we own the clock: advance the sim one fixed step.
    world_frame = world.tick()

    # bundle one synchronized snapshot: from every camera pull the image whose
    # frame matches THIS tick. a queue may briefly hold an older frame, so discard
    # until we reach world_frame; the timeout surfaces a stalled sensor rather than
    # hanging forever.
    snapshot = {}
    for name, q in image_queues.items():
        while True:
            image = q.get(timeout=2.0)
            if image.frame == world_frame:
                snapshot[name] = image
                break

    # live preview of all six cameras
    for name, image in snapshot.items():
        camera_data[name] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))
    for name in grid:
        cv2.imshow(titles[name], camera_data[name])

    # while recording: save the raw frame from every camera under a contiguous index
    # (all six share the same `seq`, so seq is the cross-camera sync key), and buffer
    # the ego state. the 3 s clip samples are assembled from this buffer afterwards.
    if recording:
        for name, image in snapshot.items():
            path = os.path.join(RUN_DIR, folder_for[name], f'{seq:06d}.png')
            image.save_to_disk(path)

        snap = world.get_snapshot()
        transform = vehicle.get_transform()
        velocity = vehicle.get_velocity()
        angular = vehicle.get_angular_velocity()   # CARLA: deg/s, +z is a right (clockwise) turn
        loc, rot = transform.location, transform.rotation
        speed_ms = math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)  # m/s

        # convert CARLA -> ROS/doc convention (x forward, y left, radians;
        # angular velocity + = left turn) so action labels and poses are consistent.
        w_ros = -math.radians(angular.z)            # rad/s, ROS sign (+ = left)
        yaw_ros = -math.radians(rot.yaw)            # rad, ROS sign
        action_label = dataset_builder.action_label_from_velocity(speed_ms, w_ros)
        frames_log.append({
            'frame_id': seq,                        # contiguous index, matches <seq>.png
            'sim_time': snap.timestamp.elapsed_seconds,
            'x': loc.x,
            'y': -loc.y,                            # CARLA y (right+) -> ROS y (left+)
            'yaw': yaw_ros,
            'v': speed_ms,
            'w': w_ros,
            'action_label': action_label,
            'map_context': get_map_context(carla_map, loc),
        })
        seq += 1

    key = cv2.waitKey(1) & 0xFF
    if key == ord('r'):
        recording = not recording
        print(('recording, ' if recording else 'paused, ') + f'{seq} frames buffered so far')
    elif key == ord('q'):
        break

cv2.destroyAllWindows()

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/plugins"
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
N

press 'r' to toggle recording, 'q' to quit
recording, 0 frames buffered so far


In [9]:
# stop the cameras (run after stopping the loop above)
for cam in cameras.values():
    cam.stop()

# restore asynchronous mode so the server keeps running on its own once we stop
# ticking (otherwise the sim freezes waiting for the next world.tick()).
traffic_manager.set_synchronous_mode(False)
settings = world.get_settings()
settings.synchronous_mode = False
settings.fixed_delta_seconds = None
world.apply_settings(settings)

print(f'recording stopped; {len(frames_log)} frames buffered. run the build cell to write output.json')

recording stopped; 428 frames buffered. run the build cell to write output.json


In [10]:
import json
importlib.reload(dataset_builder)

# Build 3-second samples (clip frames at 5 fps, key frame = clip centre) from the
# buffered per-frame log, and write them as a single output.json under the run folder.
# Fields follow the instruction document's "final dataset target" minus the
# language/trajectory blocks; action_label is kept as its own field.
samples = dataset_builder.build_samples(
    frames_log,
    run_id=RUN_ID,
    primary_cam='FRONT',        # key_frame / clip_frames reference this camera folder
    clip_sec=3.0,               # each sample covers 3 s, centred on its key frame
    sample_fps=5,               # clip frames subsampled to 5 fps
    sample_period_sec=1.0,      # emit one sample per second of driving
)

with open(os.path.join(RUN_DIR, 'output.json'), 'w') as fh:
    json.dump(samples, fh, indent=2)

print(f'built {len(samples)} 3 s samples from {len(frames_log)} frames '
      f'-> {os.path.join(RUN_DIR, "output.json")}')

built 12 3 s samples from 428 frames -> dataset/run01/output.json
